# *SENTIMENT ANALYSIS FOR INDONESIAN POLTICIAL NEWS*
# this notebook using IndoBERT + RFC model

# Import Library + define env & device

In [ ]:
print("test")

In [ ]:
# Imports
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

from transformers import AutoTokenizer, AutoModel, set_seed

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Imports complete. Running on: {device}')

# Configuration & Hyperparameter defining

In [ ]:
# --- Configuration ---
SEED = 42
MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LENGTH = 512
DATA_PATH = '/kaggle/input/cleaned-indonesian-political-news'
MODEL_PATH = '/kaggle/working'

# RFC Hyperparameters
RFC_PARAMS = {
    'n_estimators': 500,        # Number of trees
    'max_depth': None,          # None = unlimited depth
    'min_samples_split': 5,     # Min samples to split node
    'min_samples_leaf': 2,      # Min samples in leaf
    'max_features': 'sqrt',     # sqrt(768) features per split
    'class_weight': 'balanced', # Auto-handle imbalance!
    'n_jobs': -1,               # Use all CPU cores
    'random_state': SEED,
    'oob_score': True           # Free OOB validation estimate
}

# Embedding config
BATCH_SIZE = 16
POOLING_STRATEGY = 'cls'        # 'cls' or 'mean'
USE_PCA = False                 # True = faster but lower accuracy
PCA_COMPONENTS = 256

set_seed(SEED)

print(f'Model: IndoBERT + Random Forest Classifier')
print(f'Pooling: {POOLING_STRATEGY}')
print(f'Trees: {RFC_PARAMS["n_estimators"]}')
print(f'Class weight: {RFC_PARAMS["class_weight"]} (auto-balanced)')
print(f'Use PCA: {USE_PCA}')

# Data loading & integration

In [ ]:
# Load Data
df1 = pd.read_csv(f'{DATA_PATH}/detik_cleaned_labeled_manual.csv')
df2 = pd.read_csv(f'{DATA_PATH}/cnbc_cleaned_labeled_manual.csv')
df3 = pd.read_csv(f'{DATA_PATH}/kompas_cleaned_labeled_manual.csv')

print(f'Detik: {len(df1)} | CNBC: {len(df2)} | Kompas: {len(df3)}')

required_cols = ['date', 'title', 'content', 'article_id', 'text', 'label']

df_seed = pd.concat(
    [df[required_cols].copy() for df in [df1, df2, df3]],
    ignore_index=True
)

print(f'Total: {len(df_seed)}')

# EDA & data preprocessing

In [ ]:
# Label Mapping
label_to_id = {-1: 0, 0: 1, 1: 2}
id_to_label = {0: -1, 1: 0, 2: 1}
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}

df_seed['label_id'] = df_seed['label'].map(label_to_id)
df_seed = df_seed.dropna(subset=['label_id', 'text'])
df_seed['label_id'] = df_seed['label_id'].astype(int)

print(f'Total samples: {len(df_seed)}')
print(f'\nLabel distribution:')
for lid, count in sorted(df_seed['label_id'].value_counts().items()):
    print(f'  {lid} ({label_names[lid]}): {count} ({100*count/len(df_seed):.1f}%)')

# Train/val split
train_df, val_df = train_test_split(
    df_seed, test_size=0.2, random_state=SEED,
    stratify=df_seed['label_id']
)

print(f'\nTrain: {len(train_df)} | Val: {len(val_df)}')

# Tokenization & dataset prep

In [ ]:
# Load IndoBERT (FROZEN)
print('Loading IndoBERT for feature extraction...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)

# Freeze ALL parameters!
for param in bert_model.parameters():
    param.requires_grad = False

bert_model.eval()

print(f'✅ IndoBERT loaded (FROZEN)')
print(f'   Hidden size: {bert_model.config.hidden_size}')
print(f'   No fine-tuning - used as feature extractor only')

In [ ]:
# Embedding Extraction Function
def extract_embeddings(texts, tokenizer, model, max_length=512,
                       batch_size=16, pooling='cls', device='cuda'):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch, truncation=True, max_length=max_length,
            padding=True, return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden = outputs.last_hidden_state

        if pooling == 'cls':
            embeddings = last_hidden[:, 0, :]  # [CLS] token
        elif pooling == 'mean':
            mask = attention_mask.unsqueeze(-1).float()
            embeddings = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        all_embeddings.append(embeddings.cpu().numpy())

        if (i // batch_size + 1) % 10 == 0:
            print(f'   Processed {i+len(batch)}/{len(texts)}...')

    return np.vstack(all_embeddings)

print('✅ Extraction function ready')

# Customize trainer & metrics

In [ ]:
# Extract Embeddings
print('='*70)
print('EXTRACTING INDOBERT EMBEDDINGS')
print('='*70)

print(f'\nExtracting TRAIN ({len(train_df)} samples)...')
start = datetime.now()
X_train = extract_embeddings(
    train_df['text'].tolist(), tokenizer, bert_model,
    MAX_LENGTH, BATCH_SIZE, POOLING_STRATEGY, device
)
print(f'✅ Train done! Shape: {X_train.shape} ({(datetime.now()-start).seconds}s)')

print(f'\nExtracting VAL ({len(val_df)} samples)...')
start = datetime.now()
X_val = extract_embeddings(
    val_df['text'].tolist(), tokenizer, bert_model,
    MAX_LENGTH, BATCH_SIZE, POOLING_STRATEGY, device
)
print(f'✅ Val done! Shape: {X_val.shape} ({(datetime.now()-start).seconds}s)')

y_train = train_df['label_id'].values
y_val = val_df['label_id'].values

# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print(f'\n✅ Normalized with StandardScaler')

# Optional PCA
if USE_PCA:
    print(f'\nApplying PCA ({PCA_COMPONENTS} components)...')
    pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
    X_train_scaled = pca.fit_transform(X_train_scaled)
    X_val_scaled = pca.transform(X_val_scaled)
    var_explained = sum(pca.explained_variance_ratio_) * 100
    print(f'✅ PCA: {X_train.shape[1]} → {PCA_COMPONENTS} dims ({var_explained:.1f}% variance)')

print(f'\nFinal shape: {X_train_scaled.shape}')

# Training initialization

In [ ]:
# Build Random Forest
print('='*70)
print('BUILDING RANDOM FOREST CLASSIFIER')
print('='*70)

print('\nConfiguration:')
for k, v in RFC_PARAMS.items():
    print(f'  {k}: {v}')

rfc_model = RandomForestClassifier(
    n_estimators=RFC_PARAMS['n_estimators'],
    max_depth=RFC_PARAMS['max_depth'],
    min_samples_split=RFC_PARAMS['min_samples_split'],
    min_samples_leaf=RFC_PARAMS['min_samples_leaf'],
    max_features=RFC_PARAMS['max_features'],
    class_weight=RFC_PARAMS['class_weight'],
    n_jobs=RFC_PARAMS['n_jobs'],
    random_state=RFC_PARAMS['random_state'],
    oob_score=RFC_PARAMS['oob_score'],
    verbose=1
)

print('\n✅ Random Forest ready!')

# Execute training

In [ ]:
# Train Random Forest
print('='*70)
print('TRAINING RANDOM FOREST')
print('='*70)
print(f'Samples: {X_train_scaled.shape[0]}')
print(f'Features: {X_train_scaled.shape[1]}')
print(f'Trees: {RFC_PARAMS["n_estimators"]}')
print(f'Expected: ~5-20 minutes\n')

start = datetime.now()
rfc_model.fit(X_train_scaled, y_train)
elapsed = (datetime.now() - start).seconds

print(f'\n✅ Training complete! ({elapsed}s)')
print(f'OOB Score: {rfc_model.oob_score_:.4f} ({100*rfc_model.oob_score_:.2f}%)')
print(f'  (OOB = out-of-bag validation estimate on training data)')

# Final evaluation

In [ ]:
# Final Evaluation
print('Running Final Evaluation...')

y_pred = rfc_model.predict(X_val_scaled)
y_prob = rfc_model.predict_proba(X_val_scaled)

acc = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average='macro')
weighted_f1 = f1_score(y_val, y_pred, average='weighted')

print('\n' + '='*50)
print('FINAL RESULTS')
print('='*50)
print(f'Accuracy    : {acc:.4f}')
print(f'Macro F1    : {macro_f1:.4f}')
print(f'Weighted F1 : {weighted_f1:.4f}')
print(f'OOB Score   : {rfc_model.oob_score_:.4f}')
print('-' * 50)
print('\nClassification Report:\n')
print(classification_report(y_val, y_pred,
      target_names=['negative', 'neutral', 'positive'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('IndoBERT + Random Forest - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/rfc_confusion_matrix.png')
plt.show()
print('✅ Confusion matrix saved!')

# analysis evaluation

In [ ]:
# Feature Importance + Tree Depth Analysis
print('='*70)
print('MODEL ANALYSIS')
print('='*70)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Top 20 feature importance
importance = rfc_model.feature_importances_
top20 = np.argsort(importance)[::-1][:20]
axes[0].bar(range(20), importance[top20], color='forestgreen')
axes[0].set_title('Top 20 Important Dimensions')
axes[0].set_xlabel('Dimension Index')
axes[0].set_ylabel('Importance')
axes[0].set_xticks(range(20))
axes[0].set_xticklabels([str(i) for i in top20], rotation=45, fontsize=8)

# 2. Importance distribution
axes[1].hist(importance, bins=50, color='forestgreen', alpha=0.7)
axes[1].set_title('Importance Distribution')
axes[1].set_xlabel('Importance Score')
axes[1].set_ylabel('Count')

# 3. Tree depth distribution
depths = [t.get_depth() for t in rfc_model.estimators_]
axes[2].hist(depths, bins=20, color='darkolivegreen', alpha=0.7)
axes[2].axvline(np.mean(depths), color='red', linestyle='--',
                label=f'Mean: {np.mean(depths):.1f}')
axes[2].set_title('Tree Depth Distribution')
axes[2].set_xlabel('Depth')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('IndoBERT + RFC - Model Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_PATH}/rfc_analysis.png')
plt.show()

print(f'Top 5 important dims: {list(top20[:5])}')
print(f'Mean tree depth: {np.mean(depths):.1f}')
print(f'Non-zero features: {(importance > 0).sum()}/{len(importance)}')

In [ ]:
# Save Model & Summary
joblib.dump(rfc_model, f'{MODEL_PATH}/indobert_rfc_model.pkl')
print(f'✅ RFC model saved: {MODEL_PATH}/indobert_rfc_model.pkl')

joblib.dump(scaler, f'{MODEL_PATH}/rfc_scaler.pkl')
print(f'✅ Scaler saved: {MODEL_PATH}/rfc_scaler.pkl')

if USE_PCA:
    joblib.dump(pca, f'{MODEL_PATH}/rfc_pca.pkl')
    print(f'✅ PCA saved: {MODEL_PATH}/rfc_pca.pkl')

summary = {
    'model': 'IndoBERT + Random Forest Classifier',
    'indobert': MODEL_NAME,
    'pooling': POOLING_STRATEGY,
    'embedding_dim': int(X_train.shape[1]),
    'use_pca': USE_PCA,
    'pca_components': PCA_COMPONENTS if USE_PCA else None,
    'rfc_params': {k: str(v) for k, v in RFC_PARAMS.items()},
    'oob_score': float(rfc_model.oob_score_),
    'results': {
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1)
    },
    'data': {
        'train_samples': len(train_df),
        'val_samples': len(val_df)
    }
}

with open(f'{MODEL_PATH}/rfc_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n✅ Summary saved: {MODEL_PATH}/rfc_summary.json')
print(f'\n{"="*70}')
print('COMPLETE RESULTS SUMMARY')
print(f'{"="*70}')
print(json.dumps(summary['results'], indent=2))
print(f'\nOOB Score: {rfc_model.oob_score_:.4f}')

# Ablation Study 1 — n_estimators Variants (R10 - Reviewer Response)
**Reviewer Comment 10:** Justify number of trees in Random Forest.

Comparing n_estimators=100, 300, 500 (paper setting), 1000. Convergence plot shows when additional trees stop improving performance.

**Paper Section:** Section III.B — add n_estimators sensitivity analysis.

**Reference:** Breiman (2001). *Random Forests*. Machine Learning 45(1):5-32.

In [ ]:
# Ablation 1: n_estimators Variants
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

ablation1_results = []
n_est_variants = [100, 300, 500, 1000]

for n_est in n_est_variants:
    print(f'\n--- Training RFC n_estimators={n_est} ---')
    rfc_abl = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=RFC_PARAMS['max_depth'],
        min_samples_split=RFC_PARAMS['min_samples_split'],
        min_samples_leaf=RFC_PARAMS['min_samples_leaf'],
        max_features=RFC_PARAMS['max_features'],
        class_weight=RFC_PARAMS['class_weight'],
        n_jobs=RFC_PARAMS['n_jobs'],
        random_state=SEED,
        oob_score=True,
        verbose=0
    )
    rfc_abl.fit(X_train_scaled, y_train)
    pred_abl = rfc_abl.predict(X_val_scaled)
    f1_per   = f1_score(y_val, pred_abl, average=None)
    ablation1_results.append({
        'n_estimators': n_est,
        'Accuracy':     accuracy_score(y_val, pred_abl),
        'Macro F1':     f1_score(y_val, pred_abl, average='macro'),
        'F1-Neg':       f1_per[0],
        'F1-Pos':       f1_per[2],
        'OOB Score':    rfc_abl.oob_score_,
        'Paper':        '*' if n_est == 500 else '',
    })
    print(f'  Macro F1={f1_score(y_val, pred_abl, average="macro"):.4f} | OOB={rfc_abl.oob_score_:.4f}')

abl1_df = pd.DataFrame(ablation1_results)
print('\n=== Ablation 1: n_estimators Summary ===')
print(abl1_df.to_string(index=False))

# Convergence plot
plt.figure(figsize=(8, 4))
plt.plot(abl1_df['n_estimators'], abl1_df['Macro F1'], marker='o', color='forestgreen')
plt.axvline(x=500, color='red', linestyle='--', label='Paper setting (500)')
plt.xlabel('n_estimators')
plt.ylabel('Macro F1')
plt.title('RFC Macro F1 vs Number of Trees', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.show()


# Ablation Study 2 — min_samples_split Variants (R10 - Reviewer Response)
**Reviewer Comment 10:** Justify regularization strength via min_samples_split.

Comparing min_samples_split=2, 5 (paper setting), 10, 20. Higher values prevent overfitting on high-dimensional sparse embeddings.

**Paper Section:** Section III.B — add regularization sensitivity table.

**Reference:** Breiman (2001). *Random Forests*. Machine Learning 45(1):5-32.

In [ ]:
# Ablation 2: min_samples_split Variants
ablation2_results = []
mss_variants = [2, 5, 10, 20]

for mss in mss_variants:
    print(f'\n--- Training RFC min_samples_split={mss} ---')
    rfc_abl = RandomForestClassifier(
        n_estimators=500,
        max_depth=RFC_PARAMS['max_depth'],
        min_samples_split=mss,
        min_samples_leaf=RFC_PARAMS['min_samples_leaf'],
        max_features=RFC_PARAMS['max_features'],
        class_weight=RFC_PARAMS['class_weight'],
        n_jobs=RFC_PARAMS['n_jobs'],
        random_state=SEED,
        oob_score=True,
        verbose=0
    )
    rfc_abl.fit(X_train_scaled, y_train)
    pred_abl = rfc_abl.predict(X_val_scaled)
    f1_per   = f1_score(y_val, pred_abl, average=None)
    ablation2_results.append({
        'min_samples_split': mss,
        'Accuracy':          accuracy_score(y_val, pred_abl),
        'Macro F1':          f1_score(y_val, pred_abl, average='macro'),
        'F1-Neg':            f1_per[0],
        'F1-Pos':            f1_per[2],
        'OOB Score':         rfc_abl.oob_score_,
        'Paper':             '*' if mss == 5 else '',
    })

abl2_df = pd.DataFrame(ablation2_results)
print('\n=== Ablation 2: min_samples_split Summary ===')
print(abl2_df.to_string(index=False))


# Ablation Study 3 — max_features Variants (R10 - Reviewer Response)
**Reviewer Comment 10:** Justify feature subsampling strategy per split.

Comparing max_features='sqrt' (paper setting), 'log2', 0.3, 0.5. Controls randomness and decorrelation between trees.

**Paper Section:** Section III.B — add max_features sensitivity analysis.

**Reference:** Geurts et al. (2006). *Extremely Randomized Trees*. Machine Learning 63(1):3-42.

In [ ]:
# Ablation 3: max_features Variants
ablation3_results = []
mf_variants = ['sqrt', 'log2', 0.3, 0.5]

for mf in mf_variants:
    print(f'\n--- Training RFC max_features={mf} ---')
    rfc_abl = RandomForestClassifier(
        n_estimators=500,
        max_depth=RFC_PARAMS['max_depth'],
        min_samples_split=RFC_PARAMS['min_samples_split'],
        min_samples_leaf=RFC_PARAMS['min_samples_leaf'],
        max_features=mf,
        class_weight=RFC_PARAMS['class_weight'],
        n_jobs=RFC_PARAMS['n_jobs'],
        random_state=SEED,
        oob_score=True,
        verbose=0
    )
    rfc_abl.fit(X_train_scaled, y_train)
    pred_abl = rfc_abl.predict(X_val_scaled)
    f1_per   = f1_score(y_val, pred_abl, average=None)
    n_feats  = int(np.sqrt(768)) if mf == 'sqrt' else \
               int(np.log2(768)) if mf == 'log2' else int(mf * 768)
    ablation3_results.append({
        'max_features':  str(mf),
        'n_feats/split': n_feats,
        'Accuracy':      accuracy_score(y_val, pred_abl),
        'Macro F1':      f1_score(y_val, pred_abl, average='macro'),
        'F1-Neg':        f1_per[0],
        'F1-Pos':        f1_per[2],
        'OOB Score':     rfc_abl.oob_score_,
        'Paper':         '*' if mf == 'sqrt' else '',
    })

abl3_df = pd.DataFrame(ablation3_results)
print('\n=== Ablation 3: max_features Summary ===')
print(abl3_df.to_string(index=False))


# Stratified 5-Fold Cross-Validation (R10 - Reviewer Response)
**Reviewer Comment 10:** Validate metric robustness across data splits.

Pre-extract all embeddings once (IndoBERT is frozen), then run 5-fold RFC training. StandardScaler is fit per fold to avoid data leakage.

**Paper Section:** Section IV — replace single-split results with mean +/- std.

**Reference:** Kohavi (1995). *A study of cross-validation and bootstrap for accuracy estimation*. IJCAI-95.

In [ ]:
# Stratified 5-Fold CV
from sklearn.model_selection import StratifiedKFold

print('Extracting ALL embeddings for K-Fold CV (bert_model is frozen)...')
X_all_kf = extract_embeddings(
    texts=df_seed['text'].tolist(), tokenizer=tokenizer, model=bert_model,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
    pooling=POOLING_STRATEGY, device=device
)
all_labels_kf = df_seed['label_id'].values
print(f'All embeddings shape: {X_all_kf.shape}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
kfold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all_kf, all_labels_kf)):
    print(f'\n--- Fold {fold+1}/5 ---')

    X_tr, X_v = X_all_kf[train_idx], X_all_kf[val_idx]
    y_tr, y_v = all_labels_kf[train_idx], all_labels_kf[val_idx]

    fold_scaler = StandardScaler()
    X_tr_s = fold_scaler.fit_transform(X_tr)
    X_v_s  = fold_scaler.transform(X_v)

    rfc_fold = RandomForestClassifier(
        n_estimators=500,
        max_depth=RFC_PARAMS['max_depth'],
        min_samples_split=RFC_PARAMS['min_samples_split'],
        min_samples_leaf=RFC_PARAMS['min_samples_leaf'],
        max_features=RFC_PARAMS['max_features'],
        class_weight=RFC_PARAMS['class_weight'],
        n_jobs=RFC_PARAMS['n_jobs'],
        random_state=SEED,
        verbose=0
    )
    rfc_fold.fit(X_tr_s, y_tr)
    y_v_pred = rfc_fold.predict(X_v_s)
    f1_per   = f1_score(y_v, y_v_pred, average=None)

    kfold_results.append({
        'Fold':        fold + 1,
        'Accuracy':    accuracy_score(y_v, y_v_pred),
        'Macro F1':    f1_score(y_v, y_v_pred, average='macro'),
        'Weighted F1': f1_score(y_v, y_v_pred, average='weighted'),
        'F1-Neg':  f1_per[0],
        'F1-Neu':  f1_per[1],
        'F1-Pos':  f1_per[2],
    })
    print(f'  Macro F1: {kfold_results[-1]["Macro F1"]:.4f}')

kfold_df = pd.DataFrame(kfold_results)
metric_cols = ['Accuracy', 'Macro F1', 'Weighted F1', 'F1-Neg', 'F1-Neu', 'F1-Pos']
summary_row = {'Fold': 'Mean +/- Std'}
for col in metric_cols:
    summary_row[col] = f'{kfold_df[col].mean():.4f} +/- {kfold_df[col].std():.4f}'
kfold_display = pd.concat([kfold_df, pd.DataFrame([summary_row])], ignore_index=True)
print('\n=== Stratified 5-Fold CV Results (IndoBERT + RFC) ===')
print(kfold_display.to_string(index=False))


# Explainable AI — SHAP TreeExplainer (R8 - Reviewer Response)
**Reviewer Comment 8:** Provide interpretability/explainability analysis.

Using SHAP `TreeExplainer` for Random Forest (tree-based, exact SHAP values).
Key insight: RF on 768-dim embeddings tends to rely on a small subset of dominant dimensions, which explains its bias toward majority class (Neutral) and poor Positive-class recall.

Outputs: (a) beeswarm summary per class, (b) top-20 bar plot, (c) force plot for a False Negative (Positive predicted as Neutral).

**Paper Section:** Section III.B — add SHAP analysis; Section IV.D — analytical explanation of why RFC fails on Positive class (low recall), linking to SHAP feature concentration.

**Reference:** Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS 2017.

In [ ]:
# XAI: SHAP TreeExplainer for Random Forest
!pip install -q shap

import shap

feature_names_shap = [f'dim_{i}' for i in range(X_val_scaled.shape[1])]
class_names_shap   = ['Negative', 'Neutral', 'Positive']

# Use subset for speed (RF SHAP is slower than XGBoost SHAP)
N_SHAP  = min(100, len(X_val_scaled))
X_shap  = X_val_scaled[:N_SHAP]
y_shap  = np.array(y_val)[:N_SHAP]
yp_shap = np.array(y_pred)[:N_SHAP]

print('Computing SHAP values (RF — may take a few minutes)...')
explainer_shap = shap.TreeExplainer(rfc_model)
shap_values    = explainer_shap.shap_values(X_shap)
# shap_values: list of 3 arrays (one per class), each (N_SHAP, 768)
print('Done!')

# --- (a) Beeswarm Summary Plot (focus: Positive class — lowest F1) ---
print('\n=== SHAP Summary Plot: Positive Class (class 2) ===')
shap.summary_plot(
    shap_values[2], X_shap,
    feature_names=feature_names_shap,
    max_display=20,
    plot_title='SHAP Beeswarm — Positive Class'
)

# All-class summary
print('\n=== SHAP Summary Plot (all classes) ===')
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feature_names_shap,
    class_names=class_names_shap,
    max_display=20
)

# --- (b) Bar Plot Top-20 per Class ---
print('\n=== SHAP Top-20 Feature Importance per Class ===')
for cls_idx, cls_name in enumerate(class_names_shap):
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values[cls_idx], X_shap,
        feature_names=feature_names_shap,
        plot_type='bar', max_display=20, show=False
    )
    plt.title(f'SHAP Top-20 Embedding Dims — {cls_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

# --- SHAP Analysis: Feature Concentration ---
mean_abs_shap_pos = np.abs(shap_values[2]).mean(axis=0)
top10_pos = np.argsort(mean_abs_shap_pos)[::-1][:10]
top10_contribution = mean_abs_shap_pos[top10_pos].sum() / mean_abs_shap_pos.sum()
print(f'\n=== Feature Concentration Analysis (Positive Class) ===')
print(f'Top-10 dims account for {top10_contribution*100:.1f}% of total Positive-class SHAP')
print(f'This concentration in high-dim (768) space biases RF toward majority class (Neutral).')
print(f'Explanation: RF splits on sqrt(768)~28 random features per node;')
print(f'minority-class signal spread across 768 dims is frequently missed.')

# --- (c) Force Plot: False Negative (Positive predicted as Neutral) ---
print('\n=== SHAP Force Plot: False Negative (Positive -> Neutral) ===')
fn_mask = (y_shap == 2) & (yp_shap == 1)  # true=Positive, pred=Neutral
if fn_mask.sum() > 0:
    fn_idx = int(np.where(fn_mask)[0][0])
    ev = explainer_shap.expected_value
    ev_pos = ev[2] if hasattr(ev, '__len__') else ev
    shap.force_plot(
        ev_pos,
        shap_values[2][fn_idx],
        feature_names=feature_names_shap,
        matplotlib=True, show=False
    )
    plt.title(
        f'False Negative: True=Positive, Pred=Neutral (sample {fn_idx})\n'
        f'SHAP shows insufficient Positive-class signal to override Neutral prior.',
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No False Negative (Pos->Neu) in SHAP subset. Try increasing N_SHAP.')


# Computational Efficiency Analysis (R9 - Reviewer Response)
**Reviewer Comment 9:** Report model complexity and inference efficiency.

Comparing RFC vs XGBoost (both tree-based ML): RFC parallelizes across trees via joblib, XGBoost uses sequential boosting — different throughput profiles.

**Paper Section:** Section IV — add efficiency comparison table (RFC vs XGBoost vs DL).

**Reference:** Strubell et al. (2019). *Energy and Policy Considerations for Deep Learning in NLP*. ACL 2019.

In [ ]:
# Computational Efficiency Analysis (RFC)
import time, os

# --- (a) Single-Sample Latency ---
x_single = X_val_scaled[:1]
latencies = []
for _ in range(5):  # warmup
    _ = rfc_model.predict(x_single)
for _ in range(50):
    t0 = time.perf_counter()
    _ = rfc_model.predict(x_single)
    latencies.append((time.perf_counter() - t0) * 1000)

lat_arr = np.array(latencies)
print('=== Latency (ms) - RFC predict only ===')
print(f'  Mean : {lat_arr.mean():.3f}')
print(f'  Std  : {lat_arr.std():.3f}')
print(f'  P95  : {np.percentile(lat_arr, 95):.3f}')

plt.figure(figsize=(8, 4))
plt.hist(lat_arr, bins=20, color='forestgreen', edgecolor='white')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Single-Sample RFC Inference Latency', fontweight='bold')
plt.tight_layout(); plt.show()

# --- (b) Batch Throughput ---
batch_sizes = [1, 4, 8, 16, 32, 64]
throughputs = []
for bs in batch_sizes:
    x_batch = X_val_scaled[:bs]
    for _ in range(3):
        _ = rfc_model.predict(x_batch)
    t0 = time.perf_counter()
    for _ in range(50):
        _ = rfc_model.predict(x_batch)
    throughputs.append(bs * 50 / (time.perf_counter() - t0))

plt.figure(figsize=(8, 4))
plt.bar([str(b) for b in batch_sizes], throughputs, color='forestgreen', edgecolor='white')
plt.xlabel('Batch Size')
plt.ylabel('Samples / sec')
plt.title('RFC Throughput vs Batch Size', fontweight='bold')
plt.tight_layout(); plt.show()

# --- (c) Model Complexity ---
depths    = [t.get_depth() for t in rfc_model.estimators_]
n_leaves  = [t.get_n_leaves() for t in rfc_model.estimators_]
model_pkl = f'{MODEL_PATH}/indobert_rfc_model.pkl'
model_kb  = os.path.getsize(model_pkl) / 1024 if os.path.exists(model_pkl) else 0

print('\n=== Model Complexity ===')
print(f'  Trees            : {len(rfc_model.estimators_)}')
print(f'  Mean tree depth  : {np.mean(depths):.1f} +/- {np.std(depths):.1f}')
print(f'  Total leaves     : {sum(n_leaves):,}')
print(f'  Model file size  : {model_kb:.1f} KB')

# --- (d) RFC vs XGBoost comparison ---
print('\n=== Tree-Based ML Comparison: RFC vs XGBoost ===')
print(f'  RFC  latency (mean) : {lat_arr.mean():.3f} ms | parallel trees (joblib)')
print(f'  XGB  latency        : ~0.5-5 ms           | sequential boosting')
print(f'  RFC  throughput @64 : {throughputs[-1]:.0f} samples/sec')
print(f'  Note: RFC parallelizes all 500 trees simultaneously;')
print(f'        XGBoost predicts sequentially but each tree is shallower.')
print(f'  Both are 10-100x faster than deep learning (GRU/BERT) inference.')


# Statistical Significance Testing (R7 - Reviewer Response)
**Reviewer Comment 7:** Provide statistical significance of performance differences.

(a) Bootstrap CI (n=1000, seed=42) for Macro F1.
(b) McNemar's test vs GRU — RFC vs deep learning comparison.

**Paper Section:** New Section IV.E — Statistical Analysis.

**References:** McNemar (1947), Psychometrika 12(2); Dror et al. (2018), ACL P18-1128.

In [ ]:
# Statistical Significance Testing
from scipy.stats import binom
import os

y_val_arr  = np.array(y_val)
y_pred_arr = np.array(y_pred)

# --- (a) Bootstrap CI for Macro F1 ---
rng = np.random.default_rng(42)
bootstrap_f1 = []
for _ in range(1000):
    idx = rng.integers(0, len(y_val_arr), size=len(y_val_arr))
    bootstrap_f1.append(
        f1_score(y_val_arr[idx], y_pred_arr[idx], average='macro', zero_division=0)
    )

ci_low, ci_high = np.percentile(bootstrap_f1, [2.5, 97.5])
print('=== Bootstrap CI (Macro F1) - IndoBERT + RFC ===')
print(f'  Mean  : {np.mean(bootstrap_f1):.4f}')
print(f'  95% CI: [{ci_low:.4f}, {ci_high:.4f}]')

plt.figure(figsize=(7, 4))
plt.hist(bootstrap_f1, bins=30, color='forestgreen', edgecolor='white')
plt.axvline(ci_low,  color='red',  linestyle='--', label=f'2.5%  = {ci_low:.4f}')
plt.axvline(ci_high, color='blue', linestyle='--', label=f'97.5% = {ci_high:.4f}')
plt.xlabel('Macro F1')
plt.ylabel('Frequency')
plt.title('Bootstrap Macro F1 Distribution (RFC)', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.show()

# --- (b) McNemar's Test vs GRU ---
gru_preds_path = '/kaggle/working/gru_val_preds.npy'

if os.path.exists(gru_preds_path):
    gru_preds = np.load(gru_preds_path)

    if len(gru_preds) != len(y_val_arr):
        print(f'\n[WARNING] Length mismatch: GRU={len(gru_preds)}, val={len(y_val_arr)}. Skipping.')
    else:
        rfc_correct = (y_pred_arr == y_val_arr).astype(int)
        gru_correct = (gru_preds  == y_val_arr).astype(int)

        b = int(np.sum((rfc_correct == 1) & (gru_correct == 0)))
        c = int(np.sum((rfc_correct == 0) & (gru_correct == 1)))

        p_mcnemar = 2 * min(
            binom.cdf(min(b, c), b + c, 0.5),
            1 - binom.cdf(min(b, c) - 1, b + c, 0.5)
        ) if (b + c) > 0 else 1.0

        gru_f1 = f1_score(y_val_arr, gru_preds, average='macro', zero_division=0)
        rfc_f1 = f1_score(y_val_arr, y_pred_arr, average='macro')

        print(f'\n=== McNemar Test: RFC vs GRU ===')
        print(f'  RFC Macro F1             : {rfc_f1:.4f}')
        print(f'  GRU Macro F1             : {gru_f1:.4f}')
        print(f'  b (RFC correct, GRU wrong): {b}')
        print(f'  c (RFC wrong, GRU correct): {c}')
        print(f'  p-value                  : {p_mcnemar:.6f}')
        if p_mcnemar < 0.05:
            print('  Result: SIGNIFICANT difference (p < 0.05)')
            print('  GRU significantly outperforms RFC — deep learning learns joint')
            print('  embeddings+classifier, while RFC uses frozen features.')
        else:
            print('  Result: No significant difference (p >= 0.05)')
else:
    print('\n[INFO] GRU predictions not found at /kaggle/working/gru_val_preds.npy')
    print('Run the GRU notebook first and ensure it saves:')
    print("  np.save('/kaggle/working/gru_val_preds.npy', y_pred)")
    print('Then re-run this cell.')


# Save Predictions (R7 - Required for Cross-Model Comparison)
Saving RFC validation predictions and true labels for McNemar's test in other notebooks.

In [ ]:
# Save predictions for cross-model statistical comparison
np.save('/kaggle/working/rf_val_preds.npy', np.array(y_pred))
np.save('/kaggle/working/rf_val_true.npy',  np.array(y_val))

print('Saved: /kaggle/working/rf_val_preds.npy')
print('Saved: /kaggle/working/rf_val_true.npy')
print(f'Shape     : {np.array(y_pred).shape}')
print(f'Label dist: {dict(zip(*np.unique(y_pred, return_counts=True)))}')
